# CMFD Basics

Coarse-mesh finite-difference (CMFD) acceleration speeds up power iteration for k-eigenvalue
problems. After each transport update, CMFD restricts the scalar flux and the face partial currents
to a coarse mesh and coarse energy groups. It solves a small k-eigenvalue diffusion problem whose
face couplings are corrected to reproduce both transport partial currents (the partial-current, or
pCMFD, closure). It then scales the transport scalar flux of each coarse cell and coarse group by
the ratio of the new to the old coarse flux. The slowly converging fission-source modes that limit
power iteration are smooth, so the coarse problem represents them well and removes them in a few
iterations.

The method itself is fixed. The user chooses the coarse mesh (`coarse_mesh`) and the number of
coarse energy groups (`coarse_groups`) on `CMFDAcceleration`, and the convergence tolerances and the
number of transport sweeps per update on `PowerIterationKEigenSolver`.

This tutorial solves the C5G7 core with unaccelerated power iteration, with CMFD at its defaults,
and with the CMFD settings used throughout this series. The other tutorials study one group of
CMFD options each.

## The C5G7 model

The C5G7 benchmark [Lewis et al., NEA/NSC/DOC(2003)16] is a quarter core of four 17 x 17 pin
assemblies and a water reflector. The UO$_2$ assemblies sit on the diagonal. The MOX assemblies
have three enrichment zones (4.3%, 7.0% and 8.7%). Every assembly has 24 guide tubes and a central
fission chamber. The pin pitch is 1.26 cm and the reflector is one assembly (21.42 cm) thick. The
core center is in the lower-left corner, so the left and bottom boundaries are reflecting and the
right and top boundaries are vacuum. The benchmark provides seven-group cross sections with
upscattering in the four thermal groups.

The tutorials in this series use two models of that core.

* The **pin-homogenized model** mixes each fuel pin with its moderator by volume, so every pin
  cell is one 1.26 cm square mesh cell. It keeps the assembly layout, the MOX zoning, the guide
  tubes and the reflector, which are what make C5G7 hard for unaccelerated power iteration, and
  it solves in a few seconds. The option studies use it.
* The **heterogeneous model** in the [benchmark tutorial](c5g7_benchmark.ipynb) represents each
  pin by a square of the same area as the round pin, surrounded by moderator.

Both use a 2 x 2 KBA partition, so run the generated scripts with four MPI processes. A light
quadrature keeps the run times short. Production calculations need more directions.

## Model and measurement code

The next two cells build the C5G7 problem and measure a solve. They are the same in every tutorial
of this series. `solve()` builds a fresh problem for each case, so every solve starts from the
same state. Its `cmfd` argument holds the `CMFDAcceleration` options, and without it the power
iteration is unaccelerated. Its `solver` argument holds further `PowerIterationKEigenSolver`
options, such as `sweeps_per_update`. It returns the eigenvalue, the number of power iterations
(outers), the number of transport sweeps, the number of CMFD coarse cells, the number of skipped
CMFD corrections, whether the solve converged, the wall time, and the fission rate of every fuel
pin normalized to the average fuel pin. The wall time covers building the problem, initializing the
solver, and the solve. Building the problem is the same in every case.

In [ ]:
import math
from pathlib import Path

from mpi4py import MPI
from pyopensn.aquad import GLCProductQuadrature2DXY
from pyopensn.context import Finalize
from pyopensn.fieldfunc import FieldFunctionGridBased
from pyopensn.math import Vector3
from pyopensn.mesh import KBAGraphPartitioner, OrthogonalMeshGenerator
from pyopensn.solver import (
    CMFDAcceleration,
    DiscreteOrdinatesProblem,
    PowerIterationKEigenSolver,
)
from pyopensn.xs import MultiGroupXS

comm = MPI.COMM_WORLD
rank = comm.rank
tutorial_dir = Path(__file__).resolve().parent if "__file__" in globals() else Path.cwd()
xs_dir = next(
    path / "test/assets/xs/c5g7"
    for path in (tutorial_dir, *tutorial_dir.parents)
    if (path / "test/assets/xs/c5g7").is_dir()
)

In [ ]:
PITCH = 1.26  # pin pitch (cm)
PIN_RADIUS = 0.54  # fuel pin radius (cm)
ASSEMBLY_PINS = 17
CORE_PINS = 2 * ASSEMBLY_PINS  # fuel-assembly pins per side of the quarter core
DOMAIN_PINS = 3 * ASSEMBLY_PINS  # including the water reflector

WATER, UO2, MOX_43, MOX_70, MOX_87, GUIDE_TUBE, FISSION_CHAMBER = range(7)
XS_FILES = [
    "XS_water",
    "XS_UO2",
    "XS_4_3pMOX",
    "XS_7pMOX",
    "XS_8_7pMOX",
    "XS_guide_tube",
    "XS_fission_chamber",
]
FUEL = (UO2, MOX_43, MOX_70, MOX_87)
GUIDE_TUBES = {
    (2, 5), (2, 8), (2, 11), (3, 3), (3, 13), (5, 2), (5, 5), (5, 8), (5, 11), (5, 14),
    (8, 2), (8, 5), (8, 11), (8, 14), (11, 2), (11, 5), (11, 8), (11, 11), (11, 14),
    (13, 3), (13, 13), (14, 5), (14, 8), (14, 11),
}
# One quarter of the MOX assembly, mirrored about its center row and column.
# 1, 2 and 3 are 4.3%, 7.0% and 8.7% MOX, G is a guide tube and F the fission chamber.
MOX_QUARTER = [
    "111111111",
    "122222222",
    "12222G22G",
    "122G23333",
    "122233333",
    "12G33G33G",
    "122333333",
    "122333333",
    "12G33G33F",
]
MOX_ZONES = {"1": MOX_43, "2": MOX_70, "3": MOX_87}


def pin_material(px, py):
    """Return the material of the pin cell in column px and row py of the domain."""
    if px >= CORE_PINS or py >= CORE_PINS:
        return WATER
    i, j = py % ASSEMBLY_PINS, px % ASSEMBLY_PINS
    if (i, j) == (8, 8):
        return FISSION_CHAMBER
    if (i, j) in GUIDE_TUBES:
        return GUIDE_TUBE
    if px // ASSEMBLY_PINS == py // ASSEMBLY_PINS:
        return UO2
    return MOX_ZONES[MOX_QUARTER[min(i, 16 - i)][min(j, 16 - j)]]


def load_cross_sections(homogenized):
    """Load the seven C5G7 materials, mixed with water over the pin cell if homogenized."""
    cross_sections = []
    for name in XS_FILES:
        xs = MultiGroupXS()
        xs.LoadFromOpenSn(str(xs_dir / f"{name}.xs"))
        cross_sections.append(xs)
    if not homogenized:
        return cross_sections
    fuel_fraction = math.pi * PIN_RADIUS**2 / PITCH**2
    water = cross_sections[WATER]
    return [water] + [
        MultiGroupXS.Combine([(xs, fuel_fraction), (water, 1.0 - fuel_fraction)])
        for xs in cross_sections[1:]
    ]


def make_mesh(homogenized):
    """Build the quarter core on a 2 x 2 KBA partition, so run with four MPI processes."""
    if homogenized:
        nodes = [PITCH * p for p in range(DOMAIN_PINS + 1)]
    else:
        side = math.sqrt(math.pi) * PIN_RADIUS  # square pin with the area of the round one
        gap = 0.5 * (PITCH - side)
        nodes = [PITCH * p + offset for p in range(CORE_PINS) for offset in (0.0, gap, gap + side)]
        nodes += [PITCH * (CORE_PINS + 0.5 * k) for k in range(2 * ASSEMBLY_PINS + 1)]
    cut = PITCH * ASSEMBLY_PINS
    mesh = OrthogonalMeshGenerator(
        node_sets=[nodes, nodes],
        partitioner=KBAGraphPartitioner(nx=2, ny=2, xcuts=[cut], ycuts=[cut]),
    ).Execute()
    mesh.SetOrthogonalBoundaries()

    def block_id(centroid, old_block_id):
        px, py = int(centroid.x // PITCH), int(centroid.y // PITCH)
        if not homogenized:
            local_x, local_y = centroid.x - px * PITCH, centroid.y - py * PITCH
            if not (gap < local_x < gap + side and gap < local_y < gap + side):
                return WATER
        return pin_material(px, py)

    mesh.SetBlockIDFromFunction(block_id)
    return mesh


def make_problem(
    homogenized=True,
    n_azimuthal=4,
    group_ranges=((0, 6),),
    inner_linear_method="classic_richardson",
    l_max_its=4,
    adjoint=False,
    verbose=False,
):
    """Build the C5G7 quarter-core transport problem."""
    cross_sections = load_cross_sections(homogenized)
    quadrature = GLCProductQuadrature2DXY(
        n_polar=2, n_azimuthal=n_azimuthal, scattering_order=0
    )
    problem = DiscreteOrdinatesProblem(
        mesh=make_mesh(homogenized),
        num_groups=7,
        groupsets=[
            {
                "groups_from_to": group_range,
                "angular_quadrature": quadrature,
                "inner_linear_method": inner_linear_method,
                "l_max_its": l_max_its,
                "l_abs_tol": 1.0e-10,
            }
            for group_range in group_ranges
        ],
        xs_map=[{"block_ids": [m], "xs": xs} for m, xs in enumerate(cross_sections)],
        boundary_conditions=[
            {"name": "xmin", "type": "reflecting"},
            {"name": "ymin", "type": "reflecting"},
        ],
        options={
            "adjoint": adjoint,
            "use_precursors": False,
            "power_default_kappa": 1.0,
            "verbose_inner_iterations": False,
            "verbose_outer_iterations": verbose,
        },
    )
    return problem, cross_sections

In [ ]:
def pin_fission_rates(problem, cross_sections):
    """Return the fission rate of every fuel pin, normalized to the average fuel pin.

    Each pin center is the center of a rectangular cell, where the bilinear PWLD flux equals
    its cell average.
    """
    pins = [
        (px, py)
        for py in range(CORE_PINS)
        for px in range(CORE_PINS)
        if pin_material(px, py) in FUEL
    ]
    points = [Vector3((px + 0.5) * PITCH, (py + 0.5) * PITCH, 0.0) for px, py in pins]
    flux = FieldFunctionGridBased.GetPointValues(problem.GetScalarFluxFieldFunction(), points)
    rates = []
    for n, (px, py) in enumerate(pins):
        sigma_f = cross_sections[pin_material(px, py)].sigma_f
        rates.append(sum(sigma_f[g] * flux[g][n][0] for g in range(len(sigma_f))))
    mean = sum(rates) / len(rates)
    return {pin: rate / mean for pin, rate in zip(pins, rates)}


def solve(label, cmfd=None, solver=None, k_tol=1.0e-7, max_iters=400, **problem_options):
    """Build a fresh problem, solve it with power iteration, and return the metrics.

    cmfd holds the CMFDAcceleration options. Without it the power iteration is unaccelerated.
    solver holds further PowerIterationKEigenSolver options, such as sweeps_per_update.
    """
    comm.Barrier()
    start = MPI.Wtime()
    problem, cross_sections = make_problem(**problem_options)
    acceleration = None
    solver_options = {"problem": problem, "k_tol": k_tol, "max_iters": max_iters}
    solver_options.update(solver or {})
    if cmfd is not None:
        acceleration = CMFDAcceleration(problem=problem, **cmfd)
        solver_options["acceleration"] = acceleration
    power_iteration = PowerIterationKEigenSolver(**solver_options)
    power_iteration.Initialize()
    power_iteration.Execute()
    seconds = comm.allreduce(MPI.Wtime() - start, op=MPI.MAX)
    return {
        "label": label,
        "k": power_iteration.GetEigenvalue(),
        "outers": power_iteration.GetNumPowerIterations(),
        "sweeps": power_iteration.GetNumSweeps(),
        "converged": power_iteration.IsConverged(),
        "coarse_cells": acceleration.GetNumCoarseCells() if acceleration else 0,
        "skipped": acceleration.GetNumSkippedCorrections() if acceleration else 0,
        "seconds": seconds,
        "pins": pin_fission_rates(problem, cross_sections),
    }


def report(results, k_reference, pin_reference=None):
    """Print one row per solve and one stable key per metric for the regression tests."""
    if rank != 0:
        return
    header = f"{'case':<34}{'k_eff':>12}{'k error':>10}{'outers':>8}{'sweeps':>8}"
    header += f"{'coarse':>8}{'skips':>6}{'conv':>5}"
    header += f"{'pin error':>10}" if pin_reference else ""
    print(header + f"{'time (s)':>10}")
    for r in results:
        k_error = r["k"] - k_reference
        row = f"{r['label']:<34}{r['k']:>12.8f}{k_error:>10.1e}{r['outers']:>8}{r['sweeps']:>8}"
        row += f"{r['coarse_cells']:>8}{r['skipped']:>6}{int(r['converged']):>5}"
        if pin_reference:
            row += f"{max_pin_error(r, pin_reference):>10.1e}"
        print(row + f"{r['seconds']:>10.2f}")
    for r in results:
        key = r["label"].replace(" ", "_")
        print(f"{key} k-effective error={r['k'] - k_reference:.6e}")
        print(f"{key} converged={int(r['converged'])}")


def max_pin_error(result, pin_reference):
    """Largest difference of the normalized pin fission rates from the reference."""
    return max(abs(result["pins"][pin] - value) for pin, value in pin_reference.items())

## Unaccelerated power iteration

The reference eigenvalue of the pin-homogenized model with this quadrature is 1.188819461. It was
obtained with unaccelerated power iteration converged to `k_tol=1e-11` and
`fission_source_tol=1e-9`, and a CMFD solve converged to the same tolerances agrees with it to ten
digits. The tables report the error of each solve against it.

Every solve here uses `k_tol=1e-7` and the default `fission_source_tol=1e-6`. Both tests must pass:
the estimated relative error of the eigenvalue must be below `k_tol` and the estimated error of the
fission-source shape below `fission_source_tol`.

Without acceleration, each power iteration solves the within-group problem with five GMRES
iterations, which costs eight sweeps: one for the right-hand side, five for the iterations, one
for the initial residual and one to rebuild the angular flux.

In [ ]:
K_REFERENCE = 1.188819461

unaccelerated = solve("power iteration", inner_linear_method="petsc_gmres", l_max_its=5)

## CMFD with its defaults

Adding CMFD needs only the accelerator object, passed to the solver with `acceleration=`. With its
defaults, CMFD uses connected aggregates of about 32 transport cells within each MPI rank
(`coarse_mesh="local_aggregation"`), which may contain several materials. It keeps all seven energy
groups in the coarse problem (`coarse_groups="auto"` keeps every group up to 16), and it performs
one transport sweep per power iteration before each correction (`sweeps_per_update=0`, the
automatic value). CMFD sets the groupset to one iteration per transport update while the solver
runs and restores the groupset settings when the solve returns.

```python
cmfd = CMFDAcceleration(problem=problem)
solver = PowerIterationKEigenSolver(problem=problem, acceleration=cmfd, k_tol=1.0e-7)
```

In [ ]:
defaults = solve("CMFD defaults", cmfd={})

## CMFD with the settings of this series

The following settings suit this lattice and are the starting point of the other tutorials:

* a `reactor_lattice` coarse mesh with one coarse cell per pin cell (see the
  [coarse-mesh tutorial](cmfd_coarse_mesh.ipynb)). `origin` is the lower-left corner of the
  lattice, `pitch` the pin pitch and `shape` the number of pin cells along x and y. In the
  pin-homogenized model every pin cell is one transport cell, and in the heterogeneous model of the
  [benchmark tutorial](c5g7_benchmark.ipynb) the same coarse mesh groups the transport cells of
  each pin.
* `coarse_groups=2` collapses the seven groups into a fast and a thermal coarse group (see the
  [energy tutorial](cmfd_energy_groupsets.ipynb)).

The transport update stays at one sweep per power iteration (see the
[transport-update tutorial](cmfd_transport_update.ipynb)).

In [ ]:
PIN_LATTICE = {
    "type": "reactor_lattice",
    "origin": [0.0, 0.0],
    "pitch": [PITCH, PITCH],
    "shape": [DOMAIN_PINS, DOMAIN_PINS],
}
SERIES = {"coarse_mesh": PIN_LATTICE, "coarse_groups": 2}

pin_lattice = solve("CMFD pin lattice", cmfd=SERIES)
results = [unaccelerated, defaults, pin_lattice]
report(results, K_REFERENCE)

## Interpret the results

All three solves converge to the reference eigenvalue within the outer tolerance. Unaccelerated
power iteration needs 109 power iterations of eight sweeps each, 872 sweeps in all. CMFD at its
defaults performs only one sweep per power iteration. It needs 296 power iterations, more than
unaccelerated power iteration, but only a third of the sweeps. Its aggregates of about 32 pin cells
are optically thick, and CMFD warns about it when the solve starts:

```text
CMFD: 85 of 91 coarse cells are thicker than 3 transport mean free paths (largest 13.8758 along y).
CMFD with one sweep per update converges slowly on optically thick coarse cells, about 0.957187 per
update in a scattering medium at this thickness. ...
```

The [user guide](../../../userguide/cmfd_acceleration.rst#optically-thick-coarse-cells) explains this warning and its remedy. The
pin-lattice coarse mesh has coarse cells of at most about 1.3 mean free paths by the same estimate. It converges in 39
power iterations of one sweep each, 22 times fewer sweeps than unaccelerated power iteration, and it
is the fastest in wall time.

The cost of a CMFD power iteration is the transport update plus the coarse work: restriction,
homogenization, assembly and the coarse eigenvalue solve, which factorizes the coarse operator once
and then needs several back substitutions. On this small mesh the coarse work of one power
iteration takes about as long as seven sweeps, which is why the wall-time gain is smaller than the
sweep gain. On production meshes the sweeps dominate.

### Status lines

With `verbose_outer_iterations=True` in the problem options, every power iteration prints one line
for CMFD and one for power iteration:

```text
CMFD iteration = 12, k_eff = 1.1888650, status = applied
PI iteration = 13, k_eff = 1.1888650, k_eff_change = 3.78057e-04, fission_source_change = 9.21097e-04, rho_pcm = 15886.16; WGS = iteration_limit
```

CMFD numbers its corrections from zero and power iteration numbers its iterations from one, so
both lines belong to the same power iteration. `status = applied` means that the CMFD correction
was used. `k_eff_change` and `fission_source_change` are the relative change of k and the
root-mean-square relative change of the normalized cell fission sources, which the convergence
tests compare with `k_tol` and `fission_source_tol`. `rho_pcm` is the reactivity in pcm. The last
power iteration adds `status = converged`, and a final line reports the total number of sweeps.
`WGS = iteration_limit` is expected: CMFD deliberately stops each transport update after
`sweeps_per_update` sweeps. The [convergence tutorial](cmfd_convergence_diagnostics.ipynb)
explains the convergence tests and the diagnostics in detail.

In [ ]:
for result in results:
    assert result["converged"]
    assert abs(result["k"] - K_REFERENCE) < 1.0e-6
assert pin_lattice["sweeps"] < unaccelerated["sweeps"] / 10
assert pin_lattice["skipped"] == 0
if rank == 0:
    print(f"Pin-lattice CMFD sweep reduction={unaccelerated['sweeps'] / pin_lattice['sweeps']:.2f}")

## Finalize (for Jupyter Notebook only)

In script mode, PyOpenSn handles finalization automatically. In a Jupyter kernel, finalize OpenSn before MPI.

In [ ]:
if "opensn_console" not in globals():
    from IPython import get_ipython

    if get_ipython() is not None:
        Finalize()
        MPI.Finalize()